In [94]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.express as px
import warnings
warnings.filterwarnings('ignore')


In [95]:
try:
    df=pd.read_csv('winequality-red.csv',sep=';')
    print('Dataset Loaded Succesfully')

except:
    print('Dataset Not Found')


Dataset Loaded Succesfully


In [96]:
df

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality
0,7.4,0.700,0.00,1.9,0.076,11.0,34.0,0.99780,3.51,0.56,9.4,5
1,7.8,0.880,0.00,2.6,0.098,25.0,67.0,0.99680,3.20,0.68,9.8,5
2,7.8,0.760,0.04,2.3,0.092,15.0,54.0,0.99700,3.26,0.65,9.8,5
3,11.2,0.280,0.56,1.9,0.075,17.0,60.0,0.99800,3.16,0.58,9.8,6
4,7.4,0.700,0.00,1.9,0.076,11.0,34.0,0.99780,3.51,0.56,9.4,5
...,...,...,...,...,...,...,...,...,...,...,...,...
1594,6.2,0.600,0.08,2.0,0.090,32.0,44.0,0.99490,3.45,0.58,10.5,5
1595,5.9,0.550,0.10,2.2,0.062,39.0,51.0,0.99512,3.52,0.76,11.2,6
1596,6.3,0.510,0.13,2.3,0.076,29.0,40.0,0.99574,3.42,0.75,11.0,6
1597,5.9,0.645,0.12,2.0,0.075,32.0,44.0,0.99547,3.57,0.71,10.2,5


In [97]:
df.columns

Index(['fixed acidity', 'volatile acidity', 'citric acid', 'residual sugar',
       'chlorides', 'free sulfur dioxide', 'total sulfur dioxide', 'density',
       'pH', 'sulphates', 'alcohol', 'quality'],
      dtype='object')

In [98]:
px.box(df,x='fixed acidity')

In [99]:
px.violin(df,x='volatile acidity')

In [100]:
px.box(df,x='residual sugar')

In [101]:
df.isnull().sum().sort_values(ascending=False)

fixed acidity           0
volatile acidity        0
citric acid             0
residual sugar          0
chlorides               0
free sulfur dioxide     0
total sulfur dioxide    0
density                 0
pH                      0
sulphates               0
alcohol                 0
quality                 0
dtype: int64

In [102]:
X=df.drop('quality',axis=1)
y=df['quality']

In [103]:
from sklearn.preprocessing import LabelEncoder
le=LabelEncoder()
y=le.fit_transform(y)

In [104]:
df=df.drop('quality',axis=1)

In [105]:
from sklearn.preprocessing import StandardScaler,RobustScaler
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer,make_column_selector as selector
num=Pipeline([
    ('scaler',RobustScaler())
])
preprocessor=ColumnTransformer([
    ('nums',num,selector(dtype_include=np.number))
])
preprocessor

,transformers,"[('nums', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True
,force_int_remainder_cols,'deprecated'
,with_centering,True
,with_scaling,True
,quantile_range,"(25.0, ...)"


In [106]:
from xgboost import XGBClassifier
model=Pipeline([
    ('preprocessor',preprocessor),
    ('model',XGBClassifier())
])
model   

,steps,"[('preprocessor', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('nums', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [107]:
from sklearn.model_selection import GridSearchCV,train_test_split
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)


In [108]:
param_grid={
    'model__n_estimators':[500],
    'model__learning_rate':[0.3],
    'model__max_depth':[10]
}
cv=GridSearchCV(
    param_grid=param_grid ,
    estimator=model,
    n_jobs=3,
    verbose=1,
    scoring='f1_macro',
    cv=5,
    error_score='raise'
)
cv.fit(X_train,y_train)
y_pred=cv.predict(X_test)

Fitting 5 folds for each of 1 candidates, totalling 5 fits


In [109]:
from sklearn.metrics import accuracy_score,precision_score,f1_score,confusion_matrix,classification_report

print('accuracy',accuracy_score(y_test,y_pred))
print('precision',precision_score(y_test,y_pred,average='weighted'))
print('f1 score',f1_score(y_test,y_pred,average='weighted'))
print('confusion matrix',confusion_matrix(y_test,y_pred))
print('\nclassification report\n',classification_report(y_test,y_pred))



accuracy 0.684375
precision 0.6566984628779741
f1 score 0.6694901707297053
confusion matrix [[ 0  0  1  0  0  0]
 [ 0  0  7  3  0  0]
 [ 0  1 98 30  1  0]
 [ 0  0 25 97  9  1]
 [ 0  0  1 16 24  1]
 [ 0  0  0  1  4  0]]

classification report
               precision    recall  f1-score   support

           0       0.00      0.00      0.00         1
           1       0.00      0.00      0.00        10
           2       0.74      0.75      0.75       130
           3       0.66      0.73      0.70       132
           4       0.63      0.57      0.60        42
           5       0.00      0.00      0.00         5

    accuracy                           0.68       320
   macro avg       0.34      0.34      0.34       320
weighted avg       0.66      0.68      0.67       320

